# Test 3 — Cross-validation (K = 5)

All regression models at **default parameters**, evaluated with `KFold(5, shuffle=True,
random_state=42)`. This is Test 1's line-up, but scored by cross-validation instead of a
single split, so the numbers are averaged over five folds and come with a spread.

Metrics per fold, then averaged: MSE, MAE, RMSE, R², Adjusted R². Reads the shared
`preprocessed.csv`.

In [1]:
# ============================================================
#  Imports and settings  (self-contained: this notebook needs
#  nothing from the other notebooks except preprocessed.csv)
# ============================================================
import json, math, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, HistGradientBoostingRegressor,
                              AdaBoostRegressor, BaggingRegressor,
                              VotingRegressor, StackingRegressor)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import lightgbm as lgb

SEED = 42
TARGETS = ["DISINTEGRATION_TIME", "FRIABILITY", "Water absorption ratio"]

# ---- load the SAME preprocessed file every test notebook uses --------------
df = pd.read_csv("preprocessed.csv")
print("preprocessed.csv :", df.shape)

def get_X_y(frame, target):
    """Features are every column except the three targets. Rows where the target
    itself is missing cannot be scored, so they are dropped for that target only."""
    drop = [c for c in TARGETS if c in frame.columns]
    m = frame[target].notna()
    X = frame.loc[m].drop(columns=drop).astype(float)
    y = frame.loc[m, target].astype(float)
    return X, y


# ============================================================
#  RobustScaler was chosen over StandardScaler on domain grounds:
#  the data carries genuine but skewed values and a few data-entry
#  outliers, and RobustScaler uses the median and IQR, so those do
#  not distort the scaling. Only the models that need scaling get it
#  (linear family, SVR, KNN); tree ensembles are scale-invariant.
# ============================================================
def scaled(est):
    # median-impute the few cells the anomaly rules repaired to NaN, then RobustScale
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scaler", RobustScaler()), ("model", est)])

def imputed(est):
    # tree models that do not accept NaN get a median imputer only (no scaling needed)
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", est)])

def build_model(name, params=None):
    """The full regression zoo. `params` overrides defaults (used by the Optuna
    notebooks); when params is None the model runs at its default settings."""
    p = dict(params or {})
    if name == "LinearRegression":  return scaled(LinearRegression())
    if name == "Ridge":             return scaled(Ridge(random_state=SEED, **p))
    if name == "Lasso":             return scaled(Lasso(random_state=SEED, max_iter=5000, **p))
    if name == "ElasticNet":        return scaled(ElasticNet(random_state=SEED, max_iter=5000, **p))
    if name == "DecisionTree":      return imputed(DecisionTreeRegressor(random_state=SEED, **p))
    if name == "RandomForest":      return imputed(RandomForestRegressor(random_state=SEED, n_jobs=-1, **p))
    if name == "ExtraTrees":        return imputed(ExtraTreesRegressor(random_state=SEED, n_jobs=-1, **p))
    if name == "GradientBoosting":  return imputed(GradientBoostingRegressor(random_state=SEED, **p))
    if name == "HistGradientBoosting": return HistGradientBoostingRegressor(random_state=SEED, **p)
    if name == "AdaBoost":          return imputed(AdaBoostRegressor(random_state=SEED, **p))
    if name == "Bagging":           return imputed(BaggingRegressor(random_state=SEED, n_jobs=-1, **p))
    if name == "XGBoost":           return xgb.XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0, **p)
    if name == "LightGBM":          return lgb.LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1, **p)
    if name == "SVR":               return scaled(SVR(**p))
    if name == "KNN":               return scaled(KNeighborsRegressor(**p))
    if name == "VotingRegressor":
        # The paper's voting design: XGBoost + RandomForest + KNN + GradientBoosting, averaged.
        return VotingRegressor([
            ("xgb", xgb.XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0)),
            ("rf",  imputed(RandomForestRegressor(random_state=SEED, n_jobs=-1))),
            ("knn", scaled(KNeighborsRegressor(n_neighbors=2, weights="distance", metric="manhattan"))),
            ("gb",  imputed(GradientBoostingRegressor(random_state=SEED)))])
    if name == "StackingRegressor":
        return StackingRegressor(
            estimators=[
                ("xgb", xgb.XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0)),
                ("rf",  imputed(RandomForestRegressor(random_state=SEED, n_jobs=-1))),
                ("knn", scaled(KNeighborsRegressor(n_neighbors=2, weights="distance", metric="manhattan")))],
            final_estimator=imputed(GradientBoostingRegressor(random_state=SEED)), cv=5)
    raise ValueError(f"unknown model {name}")

# Every regression method we can reasonably fit ("all possible regression models").
ALL_MODELS = ["LinearRegression", "Ridge", "Lasso", "ElasticNet",
              "DecisionTree", "RandomForest", "ExtraTrees", "GradientBoosting",
              "HistGradientBoosting", "AdaBoost", "Bagging", "XGBoost", "LightGBM",
              "SVR", "KNN", "VotingRegressor", "StackingRegressor"]


# ============================================================
#  The five metrics asked for: MSE, MAE, RMSE, R2, Adjusted R2.
#  Adjusted R2 penalises R2 for the number of predictors p on n test rows:
#      1 - (1 - R2) * (n - 1) / (n - p - 1)
# ============================================================
def all_metrics(y_true, y_pred, n_features):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true); p = n_features
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    denom = (n - p - 1)
    adj = 1 - (1 - r2) * (n - 1) / denom if denom > 0 else np.nan
    return dict(MSE=mse, MAE=mean_absolute_error(y_true, y_pred),
                RMSE=math.sqrt(mse), R2=r2, Adjusted_R2=adj)

preprocessed.csv : (1982, 79)


In [2]:
# ============================================================
#  Inference-vs-actual plot, printed inline (no image file saved).
#  This is the "graph that tells the performance of inferencing
#  results with actual data" from the plan: predicted on the y-axis,
#  actual on the x-axis, with the ideal y=x line.
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def show_inference_plot(results_dict, title):
    """results_dict maps target -> (y_true, y_pred)."""
    fig, axes = plt.subplots(1, len(results_dict), figsize=(5 * len(results_dict), 4))
    if len(results_dict) == 1:
        axes = [axes]
    for ax, (target, (yt, yp)) in zip(axes, results_dict.items()):
        ax.scatter(yt, yp, s=12, alpha=.5, edgecolor="none")
        lo, hi = min(np.min(yt), np.min(yp)), max(np.max(yt), np.max(yp))
        ax.plot([lo, hi], [lo, hi], "r--", lw=1, label="ideal (y = x)")
        ax.set_xlabel("actual"); ax.set_ylabel("predicted")
        ax.set_title(f"{target}\nbest model inference", fontsize=9)
        ax.legend(fontsize=7)
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [3]:
from sklearn.model_selection import KFold, cross_val_predict

CV = KFold(n_splits=5, shuffle=True, random_state=SEED)
all_rows, best_infer = [], {}

for target in TARGETS:
    X, y = get_X_y(df, target)
    print(f"\n================  {target}  (n={len(y)})  ================")
    target_rows = []
    for name in ALL_MODELS:
        t0 = time.time()
        # score each fold, then average
        fold_metrics = []
        for tr_idx, te_idx in CV.split(X):
            model = build_model(name).fit(X.iloc[tr_idx], y.iloc[tr_idx])
            pred = model.predict(X.iloc[te_idx])
            fold_metrics.append(all_metrics(y.iloc[te_idx], pred, X.shape[1]))
        fm = pd.DataFrame(fold_metrics)
        m = dict(target=target, model=name,
                 MSE=fm.MSE.mean(), MAE=fm.MAE.mean(), RMSE=fm.RMSE.mean(),
                 R2=fm.R2.mean(), R2_std=fm.R2.std(), Adjusted_R2=fm.Adjusted_R2.mean(),
                 seconds=round(time.time() - t0, 1))
        target_rows.append(m); all_rows.append(m)
        print(f"  {name:22s} R2={m['R2']:7.3f} (+/-{m['R2_std']:.3f})  "
              f"RMSE={m['RMSE']:9.3f}  MAE={m['MAE']:8.3f}")
    best = max(target_rows, key=lambda r: r["R2"])
    # out-of-fold predictions of the best model, for the inference plot
    oof = cross_val_predict(build_model(best["model"]), X, y, cv=CV, n_jobs=-1)
    best_infer[target] = (y.to_numpy(), oof)
    print(f"  --> best: {best['model']} (mean CV R2={best['R2']:.3f})")

results_rs3 = pd.DataFrame(all_rows)


================  DISINTEGRATION_TIME  (n=1982)  ================
  LinearRegression       R2=  0.273 (+/-0.089)  RMSE=   38.257  MAE=  23.798
  Ridge                  R2=  0.274 (+/-0.089)  RMSE=   38.232  MAE=  23.734
  Lasso                  R2=  0.222 (+/-0.078)  RMSE=   39.646  MAE=  24.286
  ElasticNet             R2=  0.190 (+/-0.055)  RMSE=   40.515  MAE=  24.889
  DecisionTree           R2=  0.455 (+/-0.142)  RMSE=   32.683  MAE=  15.305
  RandomForest           R2=  0.697 (+/-0.058)  RMSE=   24.549  MAE=  12.040
  ExtraTrees             R2=  0.755 (+/-0.063)  RMSE=   21.924  MAE=  10.156
  GradientBoosting       R2=  0.597 (+/-0.090)  RMSE=   28.267  MAE=  16.635
  HistGradientBoosting   R2=  0.709 (+/-0.034)  RMSE=   24.221  MAE=  12.884
  AdaBoost               R2= -0.363 (+/-0.348)  RMSE=   51.764  MAE=  45.747
  Bagging                R2=  0.667 (+/-0.063)  RMSE=   25.728  MAE=  12.810
  XGBoost                R2=  0.683 (+/-0.064)  RMSE=   25.107  MAE=  12.616
  LightGB

### Results table (mean over 5 folds)

In [4]:
print(results_rs3[["target","model","MSE","MAE","RMSE","R2","R2_std","Adjusted_R2"]]
      .round(3).to_string(index=False))

                target                model      MSE    MAE   RMSE     R2  R2_std  Adjusted_R2
   DISINTEGRATION_TIME     LinearRegression 1469.928 23.798 38.257  0.273   0.089        0.100
   DISINTEGRATION_TIME                Ridge 1467.810 23.734 38.232  0.274   0.089        0.101
   DISINTEGRATION_TIME                Lasso 1579.862 24.286 39.646  0.222   0.078        0.036
   DISINTEGRATION_TIME           ElasticNet 1652.885 24.889 40.515  0.190   0.055       -0.002
   DISINTEGRATION_TIME         DecisionTree 1070.131 15.305 32.683  0.455   0.142        0.325
   DISINTEGRATION_TIME         RandomForest  602.933 12.040 24.549  0.697   0.058        0.625
   DISINTEGRATION_TIME           ExtraTrees  482.124 10.156 21.924  0.755   0.063        0.696
   DISINTEGRATION_TIME     GradientBoosting  802.149 16.635 28.267  0.597   0.090        0.501
   DISINTEGRATION_TIME HistGradientBoosting  588.843 12.884 24.221  0.709   0.034        0.639
   DISINTEGRATION_TIME             AdaBoost 2681.8

### Best model per target

In [5]:
best_rs3 = results_rs3.loc[results_rs3.groupby("target")["R2"].idxmax()]
print(best_rs3[["target","model","RMSE","R2","R2_std","Adjusted_R2"]].round(3).to_string(index=False))

                target      model   RMSE    R2  R2_std  Adjusted_R2
   DISINTEGRATION_TIME ExtraTrees 21.924 0.755   0.063        0.696
            FRIABILITY        SVR  0.930 0.047   0.084       -0.179
Water absorption ratio ExtraTrees  9.676 0.945   0.019        0.932


### Inference vs actual (out-of-fold predictions of best model)

In [6]:
show_inference_plot(best_infer, "Test 3 (CV k=5) — out-of-fold predicted vs actual")

### Save this result set

In [7]:
results_rs3.to_csv("results_test3_cv.csv", index=False)
print("saved results_test3_cv.csv")

saved results_test3_cv.csv


In [8]:
# ============================================================
#  ACTUAL vs PREDICTED — FULL preprocessed dataset (all rows)
#  Reusable across Test 1, 2, 3, 4. Change the two settings.
# ============================================================
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_predict

# ---- SETTINGS: change these two lines per notebook --------------------------
RESULTS_CSV       = "/content/results_test3_cv.csv"   # -> _test2_optuna / _test3_cv / _test4_cv_optuna
PREPROCESSED_CSV  = "/content/preprocessed.csv"               # the full cleaned dataset, all 1982 rows
MODE               = "cv"   # "holdout" for Test 1 & Test 2  |  "cv" for Test 3 & Test 4

results = pd.read_csv(RESULTS_CSV)
full_df = pd.read_csv(PREPROCESSED_CSV)   # read fresh — independent of whatever `df` is in memory


def get_best_row(results, target):
    sub = results[results.target == target]
    return sub.loc[sub.R2.idxmax()]


def summarise(actual, predicted, n_features):
    """MSE, MAE, RMSE, R2, Adjusted R2 on whatever subset is passed in."""
    n = len(actual)
    r2 = 1 - ((actual - predicted) ** 2).sum() / ((actual - actual.mean()) ** 2).sum()
    mse = ((actual - predicted) ** 2).mean()
    mae = (actual - predicted).abs().mean()
    denom = n - n_features - 1
    adj_r2 = 1 - (1 - r2) * (n - 1) / denom if denom > 0 else np.nan
    return dict(n=n, MSE=mse, MAE=mae, RMSE=np.sqrt(mse), R2=r2, Adjusted_R2=adj_r2)


def metrics_on_full_dataset(target, mode=MODE):
    row = get_best_row(results, target)
    model_name = row["model"]
    params = json.loads(row["params"]) if ("params" in row and pd.notna(row.get("params"))) else {}

    X, y = get_X_y(full_df, target)
    n_feat = X.shape[1]

    if mode == "holdout":
        # Same split the test notebook used, so the model is the same one that was reported.
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=SEED)
        model = build_model(model_name, params).fit(X_tr, y_tr)
        pred = model.predict(X)                       # predict EVERY row in preprocessed.csv
        split_label = np.where(X.index.isin(X_tr.index), "train", "test")
    elif mode == "cv":
        # cross_val_predict already gives an out-of-fold prediction for every row —
        # this IS the honest full-dataset version, no train/test contamination.
        cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
        pred = cross_val_predict(build_model(model_name, params), X, y, cv=cv, n_jobs=-1)
        split_label = np.array(["out-of-fold"] * len(X))
    else:
        raise ValueError("mode must be 'holdout' or 'cv'")

    out = pd.DataFrame({"row_index": X.index, "target": target, "model": model_name,
                        "split": split_label, "actual": y.to_numpy(), "predicted": pred})
    out["abs_error"] = (out.actual - out.predicted).abs()
    out["pct_error"] = 100 * out.abs_error / out.actual.replace(0, np.nan)

    full_m = summarise(out.actual, out.predicted, n_feat)
    if mode == "holdout":
        train_m = summarise(out.loc[out.split == "train", "actual"], out.loc[out.split == "train", "predicted"], n_feat)
        test_m  = summarise(out.loc[out.split == "test",  "actual"], out.loc[out.split == "test",  "predicted"], n_feat)
    else:
        train_m = test_m = None   # no train/test split to break out under CV

    return out, full_m, train_m, test_m


all_tables, metric_rows = [], []
for t in TARGETS:
    tbl, full_m, train_m, test_m = metrics_on_full_dataset(t)
    print(f"\n{t}  ({tbl.model.iloc[0]})  —  first 10 rows")
    print(tbl.head(10).round(3).to_string(index=False))

    rows_to_show = [("FULL preprocessed.csv", full_m)]
    if train_m is not None:
        rows_to_show += [("train rows only", train_m), ("test rows only", test_m)]
    for label, m in rows_to_show:
        print(f"  {label:24s} n={m['n']:4d}  MSE={m['MSE']:.3f}  MAE={m['MAE']:.3f}  "
              f"RMSE={m['RMSE']:.3f}  R2={m['R2']:.3f}  AdjR2={m['Adjusted_R2']:.3f}")
        metric_rows.append(dict(target=t, model=tbl.model.iloc[0], subset=label, **m))
    all_tables.append(tbl)

actual_vs_pred_full = pd.concat(all_tables, ignore_index=True)
metrics_summary = pd.DataFrame(metric_rows)

out1 = RESULTS_CSV.replace("results_", "actual_vs_predicted_FULLDATA_")
out2 = RESULTS_CSV.replace("results_", "metrics_FULLDATA_")
actual_vs_pred_full.to_csv(out1, index=False)
metrics_summary.to_csv(out2, index=False)
print(f"\nsaved {out1}  ({len(actual_vs_pred_full)} rows)")
print(f"saved {out2}")

# Display results in the terminal/output cell without saving CSV files
# print("\n" + "=" * 100)
# print("ACTUAL VS PREDICTED — FULL DATASET")
# print("=" * 100)
# print(actual_vs_pred_full.round(3).to_string(index=False))

# print("\n" + "=" * 100)
# print("METRICS SUMMARY")
# print("=" * 100)
# print(metrics_summary.round(3).to_string(index=False))


DISINTEGRATION_TIME  (ExtraTrees)  —  first 10 rows
 row_index              target      model       split  actual  predicted  abs_error  pct_error
         0 DISINTEGRATION_TIME ExtraTrees out-of-fold     9.5     22.132     12.632    132.967
         1 DISINTEGRATION_TIME ExtraTrees out-of-fold   372.0    170.373    201.627     54.201
         2 DISINTEGRATION_TIME ExtraTrees out-of-fold   190.0    178.640     11.359      5.979
         3 DISINTEGRATION_TIME ExtraTrees out-of-fold   102.0    247.300    145.300    142.451
         4 DISINTEGRATION_TIME ExtraTrees out-of-fold   252.0    297.710     45.710     18.139
         5 DISINTEGRATION_TIME ExtraTrees out-of-fold   218.0    253.560     35.560     16.312
         6 DISINTEGRATION_TIME ExtraTrees out-of-fold   307.0    242.720     64.280     20.938
         7 DISINTEGRATION_TIME ExtraTrees out-of-fold   280.0    267.510     12.490      4.461
         8 DISINTEGRATION_TIME ExtraTrees out-of-fold    16.0     16.412      0.412      2.5

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# SETTINGS
# ============================================================
RESULTS_CSV = "/content/actual_vs_predicted_FULLDATA_test3_cv.csv"
MODE = "cv"   # "holdout" for Test 1/2, "cv" for Test 3/4

# Automatically find the prediction CSV created by your code
PREDICTIONS_CSV = RESULTS_CSV.replace(
    "results_",
    "actual_vs_predicted_FULLDATA_"
)

# Read predictions
actual_vs_pred_full = pd.read_csv(PREDICTIONS_CSV)

print("File successfully loaded:", PREDICTIONS_CSV)
print("Shape:", actual_vs_pred_full.shape)
display(actual_vs_pred_full.head())


# ============================================================
# PREPARE DATA FOR PLOTTING
# ============================================================
if MODE == "holdout":
    # Only unseen test rows
    plot_df = actual_vs_pred_full[
        actual_vs_pred_full["split"] == "test"
    ].copy()
else:
    # All predictions are out-of-fold under CV
    plot_df = actual_vs_pred_full.copy()

plot_df["residual"] = (
    plot_df["actual"] - plot_df["predicted"]
)

print("\nRows being plotted:", len(plot_df))
print("Targets:", plot_df["target"].unique())


# ============================================================
# CREATE GRAPHS
# ============================================================
sns.set_theme(style="whitegrid")

for target in plot_df["target"].unique():

    d = plot_df[plot_df["target"] == target].copy()

    if d.empty:
        print(f"No rows available for target: {target}")
        continue

    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    fig.suptitle(
        f"Target: {target}\nModel: {d['model'].iloc[0]}",
        fontsize=16,
        fontweight="bold"
    )

    # 1. Actual vs predicted scatter plot
    sns.scatterplot(
        data=d,
        x="actual",
        y="predicted",
        alpha=0.7,
        ax=axes[0, 0]
    )

    minimum = min(d["actual"].min(), d["predicted"].min())
    maximum = max(d["actual"].max(), d["predicted"].max())

    axes[0, 0].plot(
        [minimum, maximum],
        [minimum, maximum],
        "r--",
        linewidth=2,
        label="Perfect prediction"
    )

    axes[0, 0].set_title("Actual vs Predicted")
    axes[0, 0].legend()

    # 2. Residual plot
    sns.scatterplot(
        data=d,
        x="predicted",
        y="residual",
        alpha=0.7,
        ax=axes[0, 1]
    )

    axes[0, 1].axhline(0, color="red", linestyle="--")
    axes[0, 1].set_title("Residual Plot")
    axes[0, 1].set_ylabel("Actual − Predicted")

    # 3. Residual distribution
    sns.histplot(
        data=d,
        x="residual",
        bins=30,
        kde=True,
        ax=axes[1, 0]
    )

    axes[1, 0].axvline(0, color="red", linestyle="--")
    axes[1, 0].set_title("Residual Distribution")

    # 4. Sorted actual vs predicted
    d_sorted = d.sort_values("actual").reset_index(drop=True)

    axes[1, 1].plot(
        d_sorted.index,
        d_sorted["actual"],
        label="Actual",
        color="blue",
        linewidth=2
    )

    axes[1, 1].plot(
        d_sorted.index,
        d_sorted["predicted"],
        label="Predicted",
        color="orange",
        linewidth=2
    )

    axes[1, 1].set_title("Sorted Actual vs Predicted")
    axes[1, 1].set_xlabel("Observations")
    axes[1, 1].legend()

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()

File successfully loaded: /content/actual_vs_predicted_FULLDATA_test3_cv.csv
Shape: (5946, 8)


,row_index,target,model,split,actual,predicted,abs_error,pct_error
0,0,DISINTEGRATION_TIME,ExtraTrees,out-of-fold,9.5,22.1319,12.6319,132.967368
1,1,DISINTEGRATION_TIME,ExtraTrees,out-of-fold,372.0,170.3734,201.6266,54.200699
2,2,DISINTEGRATION_TIME,ExtraTrees,out-of-fold,190.0,178.6405,11.3595,5.978684
3,3,DISINTEGRATION_TIME,ExtraTrees,out-of-fold,102.0,247.3000,145.3000,142.450980
4,4,DISINTEGRATION_TIME,ExtraTrees,out-of-fold,252.0,297.7100,45.7100,18.138889



Rows being plotted: 5946
Targets: ['DISINTEGRATION_TIME' 'FRIABILITY' 'Water absorption ratio']
